In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
catalog_name = "Verse_commonwealth_bank_catalog"

bronze_schema = "bronze"
silver_schema = "silver"
quarantine_schema = "quarantine"

spark.sql(f"USE CATALOG {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quarantine_schema}")

print("ETL configuration completed")

In [0]:
datasets = [
    "customers",
    "accounts",
    "transactions",
    "loans",
    "branches"
]

In [0]:
bronze_tables = {}

for dataset in datasets:

    table_name = f"{catalog_name}.{bronze_schema}.{dataset}"

    if not spark.catalog.tableExists(table_name):
        raise Exception(f"Bronze table not found: {table_name}")

    df = spark.table(table_name)

    bronze_tables[dataset] = df

    print(f"{dataset}: {df.count()} records")

In [0]:
def clean_data(df, dataset):

    # Trim string columns
    for column, dtype in df.dtypes:

        if dtype == "string":
            df = df.withColumn(
                column,
                F.trim(F.col(column))
            )

            df = df.withColumn(
                column,
                F.when(
                    F.col(column) == "",
                    None
                ).otherwise(F.col(column))
            )

    # Date columns
    date_columns = {
        "customers": ["date_of_birth", "created_date"],
        "accounts": ["open_date"],
        "transactions": ["transaction_date"],
        "loans": ["disbursement_date"],
        "branches": ["opened_date"]
    }

    for column in date_columns[dataset]:

        if column in df.columns:
            df = df.withColumn(
                column,
                F.to_date(F.col(column))
            )

    # Numeric columns
    numeric_columns = {
        "customers": [],
        "accounts": ["current_balance"],
        "transactions": ["amount"],
        "loans": [
            "principal_amount",
            "interest_rate_pct",
            "tenure_months",
            "outstanding_amount"
        ],
        "branches": []
    }

    for column in numeric_columns[dataset]:

        if column in df.columns:
            df = df.withColumn(
                column,
                F.col(column).cast("double")
            )

    # Standardize status values
    status_columns = [
        "customer_status",
        "account_status",
        "transaction_status",
        "loan_status",
        "branch_status"
    ]

    for column in status_columns:

        if column in df.columns:
            df = df.withColumn(
                column,
                F.upper(F.col(column))
            )

    return df

In [0]:
primary_keys = {
    "customers": "customer_id",
    "accounts": "account_id",
    "transactions": "transaction_id",
    "loans": "loan_id",
    "branches": "branch_id"
}


def remove_duplicates(df, dataset):

    primary_key = primary_keys[dataset]

    # Prefer the freshest record; fall back to the primary key
    # (deterministic) when no timestamp/batch column exists
    if "updated_at" in df.columns:
        order_column = F.col("updated_at").desc_nulls_last()
    elif "batch_id" in df.columns:
        order_column = F.col("batch_id").desc()
    elif "ingested_at" in df.columns:
        order_column = F.col("ingested_at").desc_nulls_last()
    else:
        order_column = F.col(primary_key).desc()

    window_spec = (
        Window
        .partitionBy(primary_key)
        .orderBy(order_column)
    )

    df = (
        df
        .withColumn(
            "_duplicate_rank",
            F.row_number().over(window_spec)
        )
        .filter(F.col("_duplicate_rank") == 1)
        .drop("_duplicate_rank")
    )

    return df

In [0]:
def validate_data(df, dataset):

    rules = []

    if dataset == "customers":

        rules = [
            (F.col("customer_id").isNotNull(),
             "Missing customer_id"),

            (F.col("first_name").isNotNull(),
             "Missing first_name"),

            (F.col("email").rlike(
                r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            ),
             "Invalid email")
        ]

    elif dataset == "accounts":

        rules = [
            (F.col("account_id").isNotNull(),
             "Missing account_id"),

            (F.col("customer_id").isNotNull(),
             "Missing customer_id"),

            (F.col("current_balance").isNotNull(),
             "Invalid current_balance"),

            (F.col("current_balance") >= 0,
             "Negative current_balance")
        ]

    elif dataset == "transactions":

        rules = [
            (F.col("transaction_id").isNotNull(),
             "Missing transaction_id"),

            (F.col("account_id").isNotNull(),
             "Missing account_id"),

            (F.col("amount").isNotNull(),
             "Missing or invalid amount"),

            (F.col("amount") > 0,
             "Amount must be positive"),

            # TODO: add an isin() enum check once the distinct
            # transaction_status values are confirmed
            (F.col("transaction_status").isNotNull(),
             "Missing transaction_status")
        ]

    elif dataset == "loans":

        rules = [
            (F.col("loan_id").isNotNull(),
             "Missing loan_id"),

            (F.col("customer_id").isNotNull(),
             "Missing customer_id"),

            (F.col("principal_amount") > 0,
             "Invalid principal amount"),

            (F.col("interest_rate_pct") >= 0,
             "Negative interest rate"),

            (F.col("outstanding_amount") >= 0,
             "Invalid outstanding amount")
        ]

    elif dataset == "branches":

        rules = [
            (F.col("branch_id").isNotNull(),
             "Missing branch_id"),

            (F.col("branch_name").isNotNull(),
             "Missing branch_name")
        ]

    # Construct rejection reasons (null-safe: a null rule
    # result counts as a failed check)
    reason_columns = [
        F.when(
            ~F.coalesce(condition, F.lit(False)),
            F.lit(reason)
        )
        for condition, reason in rules
    ]

    df = df.withColumn(
        "rejection_reason",
        F.concat_ws("; ", *reason_columns)
    )

    return (
        df.withColumn(
            "_is_valid",
            F.col("rejection_reason") == ""
        )
    )

In [0]:
def add_fk_validation(
    df,
    source_column,
    reference_df,
    reference_column,
    error_message
):

    ref = (
        reference_df
        .select(reference_column)
        .filter(F.col(reference_column).isNotNull())
        .distinct()
        .withColumn("_ref_exists", F.lit(True))
    )

    df = df.join(
        ref,
        df[source_column] == ref[reference_column],
        "left"
    )

    fk_invalid = (
        F.col(source_column).isNull() |
        F.col("_ref_exists").isNull()
    )

    df = (
        df
        .withColumn(
            "rejection_reason",
            F.when(
                fk_invalid,
                F.concat_ws(
                    "; ",
                    F.col("rejection_reason"),
                    F.lit(error_message)
                )
            ).otherwise(F.col("rejection_reason"))
        )
        .withColumn(
            "_is_valid",
            F.col("_is_valid") & ~fk_invalid
        )
        .drop("_ref_exists", reference_column)
    )

    return df

In [0]:
def load_quarantine(df, dataset):

    quarantine_table = (
        f"{catalog_name}.{quarantine_schema}.{dataset}_rejected"
    )

    rejected_df = (
        df
        .filter(~F.col("_is_valid"))
        .withColumn(
            "etl_timestamp",
            F.current_timestamp()
        )
        .drop("_is_valid")
    )

    if rejected_df.limit(1).count() > 0:

        if not spark.catalog.tableExists(quarantine_table):

            (
                rejected_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(quarantine_table)
            )

        else:

            target = DeltaTable.forName(
                spark,
                quarantine_table
            )

            merge_key = primary_keys[dataset]

            (
                target.alias("t")
                .merge(
                    rejected_df.alias("s"),
                    f"t.{merge_key} = s.{merge_key}"
                )
                .whenNotMatchedInsertAll()
                .execute()
            )

    print(f"Quarantine completed: {dataset}")

In [0]:
def load_silver(df, dataset):

    silver_table = (
        f"{catalog_name}.{silver_schema}.{dataset}"
    )

    primary_key = primary_keys[dataset]

    clean_df = (
        df
        .filter(F.col("_is_valid"))
        .drop(
            "_is_valid",
            "rejection_reason"
        )
        .withColumn(
            "etl_timestamp",
            F.current_timestamp()
        )
    )

    if clean_df.limit(1).count() == 0:

        print(f"No valid records for {dataset}")
        return

    if not spark.catalog.tableExists(silver_table):

        (
            clean_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        print(f"Created Silver table: {silver_table}")

    else:

        target = DeltaTable.forName(
            spark,
            silver_table
        )

        # Use source update timestamp where available.
        if "updated_at" in clean_df.columns:

            update_condition = """
                s.updated_at >= t.updated_at
            """

        elif "batch_id" in clean_df.columns:

            update_condition = """
                s.batch_id >= t.batch_id
            """

        else:

            update_condition = """
                s.etl_timestamp >= t.etl_timestamp
            """

        (
            target.alias("t")
            .merge(
                clean_df.alias("s"),
                f"t.{primary_key} = s.{primary_key}"
            )
            .whenMatchedUpdateAll(
                condition=update_condition
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Incremental Silver load completed: {dataset}")

In [0]:
def add_fk_validation(
    df,
    source_column,
    reference_df,
    reference_column,
    error_message
):

    ref = (
        reference_df
        .select(F.col(reference_column).alias("_ref_key"))
        .filter(F.col("_ref_key").isNotNull())
        .distinct()
        .withColumn("_ref_exists", F.lit(True))
    )

    df = df.join(
        ref,
        F.col(source_column) == F.col("_ref_key"),
        "left"
    )

    fk_invalid = (
        F.col(source_column).isNull() |
        F.col("_ref_exists").isNull()
    )

    df = (
        df
        .withColumn(
            "rejection_reason",
            F.when(
                fk_invalid,
                F.concat_ws(
                    "; ",
                    F.col("rejection_reason"),
                    F.lit(error_message)
                )
            ).otherwise(F.col("rejection_reason"))
        )
        .withColumn(
            "_is_valid",
            F.col("_is_valid") & ~fk_invalid
        )
        .drop("_ref_key", "_ref_exists")
    )

    return df

In [0]:
# Referential integrity checks (from the source data README):
# accounts/loans -> customers and branches; transactions -> accounts and customers
fk_rules = {
    "accounts": [
        ("customer_id", "customers", "customer_id",
         "Invalid customer_id reference"),
        ("branch_id", "branches", "branch_id",
         "Invalid branch_id reference"),
    ],
    "loans": [
        ("customer_id", "customers", "customer_id",
         "Invalid customer_id reference"),
        ("branch_id", "branches", "branch_id",
         "Invalid branch_id reference"),
    ],
    "transactions": [
        ("account_id", "accounts", "account_id",
         "Invalid account_id reference"),
        ("customer_id", "customers", "customer_id",
         "Invalid customer_id reference"),
    ],
}

# Masters first, then dependents — FK checks reuse the
# already-processed reference frames
processing_order = [
    "customers",
    "branches",
    "accounts",
    "loans",
    "transactions"
]

print("FK rules and processing order configured")

In [0]:
processed = {}  # cleaned + deduped + valid frames, reused for FK checks
summary = []

for dataset in processing_order:

    df = bronze_tables[dataset]
    df = clean_data(df, dataset)
    df = remove_duplicates(df, dataset)
    df = validate_data(df, dataset)

    for source_column, ref_dataset, ref_column, message in fk_rules.get(dataset, []):
        df = add_fk_validation(
            df,
            source_column,
            processed[ref_dataset],
            ref_column,
            message
        )

    total = df.count()
    valid = df.filter(F.col("_is_valid")).count()
    rejected = total - valid

    load_quarantine(df, dataset)
    load_silver(df, dataset)

    processed[dataset] = (
        df
        .filter(F.col("_is_valid"))
        .drop("_is_valid", "rejection_reason")
        .cache()
    )

    summary.append((dataset, total, valid, rejected))
    print(f"{dataset}: {total} rows -> {valid} silver, {rejected} quarantined")

display(
    spark.createDataFrame(
        summary,
        schema="dataset string, after_dedupe long, silver long, quarantined long"
    )
)

In [0]:
processed = {}

# Process branches first
for dataset in ["branches", "customers"]:

    df = bronze_tables[dataset]

    df = clean_data(df, dataset)

    df = remove_duplicates(df, dataset)

    df = validate_data(df, dataset)

    processed[dataset] = df

    load_quarantine(df, dataset)

    load_silver(df, dataset)


# Process accounts
df = clean_data(bronze_tables["accounts"], "accounts")

df = remove_duplicates(df, "accounts")

df = validate_data(df, "accounts")

df = add_fk_validation(
    df,
    "customer_id",
    processed["customers"].filter("_is_valid"),
    "customer_id",
    "Customer does not exist"
)

df = add_fk_validation(
    df,
    "branch_id",
    processed["branches"].filter("_is_valid"),
    "branch_id",
    "Branch does not exist"
)

processed["accounts"] = df

load_quarantine(df, "accounts")
load_silver(df, "accounts")


# Process loans
df = clean_data(bronze_tables["loans"], "loans")

df = remove_duplicates(df, "loans")

df = validate_data(df, "loans")

df = add_fk_validation(
    df,
    "customer_id",
    processed["customers"].filter("_is_valid"),
    "customer_id",
    "Customer does not exist"
)

df = add_fk_validation(
    df,
    "branch_id",
    processed["branches"].filter("_is_valid"),
    "branch_id",
    "Branch does not exist"
)

processed["loans"] = df

load_quarantine(df, "loans")
load_silver(df, "loans")


# Process transactions
df = clean_data(
    bronze_tables["transactions"],
    "transactions"
)

df = remove_duplicates(df, "transactions")

df = validate_data(df, "transactions")

df = add_fk_validation(
    df,
    "account_id",
    processed["accounts"].filter("_is_valid"),
    "account_id",
    "Account does not exist"
)

processed["transactions"] = df

load_quarantine(df, "transactions")
load_silver(df, "transactions")

print("All banking ETL datasets processed successfully")

In [0]:
for dataset in datasets:

    table_name = (
        f"{catalog_name}.{silver_schema}.{dataset}"
    )

    print(f"Silver table: {table_name}")

    df = spark.table(table_name)

    print("Total records:", df.count())

    display(
        df.groupBy("batch_id")
        .count()
        .orderBy("batch_id")
    )

In [0]:
for dataset in datasets:

    table_name = (
        f"{catalog_name}.{quarantine_schema}.{dataset}_rejected"
    )

    if spark.catalog.tableExists(table_name):

        print(f"Rejected records: {dataset}")

        display(
            spark.table(table_name)
            .select(
                "batch_id",
                "rejection_reason"
            )
            .groupBy(
                "batch_id",
                "rejection_reason"
            )
            .count()
        )